<h1 align="center">Join Submission and Text Data Sets to Find Narrative Facts</h1>

### Import Code Libraries

In [ ]:
import polars as pl
from IPython.display import display, HTML

display(HTML("<style>.container { width:100% !important; }</style>"))

### Read Submissions and Text Data Sets for Feb. 2024 from [Financial Statement and Notes Data Sets](https://www.sec.gov/dera/data/financial-statement-and-notes-data-set)

In [ ]:
submissions = pl.read_csv('data/2024q1_notes/sub.tsv', separator='\t', infer_schema_length=10000)
textFacts = pl.read_csv('data/2024q1_notes/txt.tsv', separator='\t', infer_schema_length=10000)

### Look at the Contents of submissions DataFrame

In [ ]:
submissions

### Look at the Contents of textFacts DataFrame

In [ ]:
textFacts

### Join submissions DataFrame with textFacts DataFrame Using adsh Column as a Join Key

In [ ]:
mergedDF = submissions.join(textFacts, on='adsh')

### Look at the Contents of mergedDF DataFrame

In [ ]:
mergedDF

### Filter for Significant Accounting Policies for Marriott International using SignificantAccountingPoliciesTextBlock tag

In [ ]:
textFactsForMarriott = (
    mergedDF
    .filter(
        (pl.col('name') == 'MARRIOTT INTERNATIONAL INC /MD/')
        & (pl.col('tag') == 'SignificantAccountingPoliciesTextBlock')
    )
    .select(['name', 'tag', 'value'])
)
textFactsForMarriott

### Filter for Exchange Act Section 12(g) Securities

In [ ]:
factsFor12gSecurities = (
    mergedDF
    .filter(pl.col('tag') == 'Security12gTitle')
    .select(['name', 'cik', 'form', 'period', 'fy', 'fp', 'filed', 'tag', 'value'])
    .sort(['name', 'value'])
)
factsFor12gSecurities

### Create an Excel File for Section 12(g) Securities

In [ ]:
# Requires openpyxl: pip install openpyxl
factsFor12gSecurities.write_excel('data/Section12gSecurities.xlsx')